Student Name : Eaint Taryar Linlat

# Task 16 - Agentic AI: Adding a P/E Ratio Tool to a Financial Agent

## Section 1 - Install Libraries and Configure Gemini API

We install `google-generativeai` (the Gemini SDK) and `yfinance` (Yahoo Finance
wrapper). The API key is loaded from Colab Secrets — never hardcode keys in notebooks.

In [ ]:
!pip install -q -U google-generativeai yfinance

import google.generativeai as genai
from google.colab import userdata
import os

# Load key from Colab Secrets (Key icon in left sidebar -> add GEMINI_API_KEY)
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print('Setup complete. Gemini API configured.')

## Section 2 - Define the Three Agent Tools

In agentic AI, a **tool** is simply a Python function with a clear docstring.
The Gemini SDK reads the docstring to understand *when* and *how* to call the
function — so good documentation is not optional, it is functional.

### Tool Architecture

| Tool | Input | Output | Purpose |
|------|-------|--------|---------|
| `get_stock_price` | ticker | float | Live price from Yahoo Finance |
| `get_company_risk_score` | ticker | dict | Beta + risk category |
| `get_pe_ratio` (**NEW**) | ticker | dict | Trailing P/E + valuation signal |

The P/E ratio tool is the core addition for Task 16. It fetches `trailingPE`
from Yahoo Finance and includes a pre-computed valuation signal so the agent
has an explicit numeric anchor to reason against the market average of 25.

In [ ]:
import yfinance as yf

# ─────────────────────────────────────────────────────────────────────────────
# Tool 1: Live Stock Price
# ─────────────────────────────────────────────────────────────────────────────
def get_stock_price(ticker: str):
    """
    Retrieves the current live stock price for a given ticker symbol.
    Use this whenever the user asks for the current or latest price of a stock.
    Args:
        ticker: The stock ticker symbol, e.g. 'AAPL' for Apple, 'MSFT' for Microsoft.
    Returns:
        The current stock price as a float, rounded to 2 decimal places.
    """
    print(f'  [TOOL] get_stock_price called for: {ticker}')
    try:
        stock = yf.Ticker(ticker)
        price = stock.fast_info['last_price']
        return round(price, 2)
    except Exception as e:
        return f'Error fetching price for {ticker}: {e}'


# ─────────────────────────────────────────────────────────────────────────────
# Tool 2: Risk Score via Beta
# ─────────────────────────────────────────────────────────────────────────────
def get_company_risk_score(ticker: str):
    """
    Calculates a risk assessment for a stock based on its Beta coefficient.
    Beta measures how much the stock moves relative to the overall market.
    Beta > 1.5 = High Risk, Beta < 0.8 = Low Risk, otherwise Moderate.
    Use this when the user asks about volatility, risk, or stability of a stock.
    Args:
        ticker: The stock ticker symbol.
    Returns:
        A dict with 'beta' (float) and 'assessment' (string risk category).
    """
    print(f'  [TOOL] get_company_risk_score called for: {ticker}')
    try:
        stock = yf.Ticker(ticker)
        beta = stock.info.get('beta', 0)
        if beta > 1.5:
            assessment = 'High Risk (High Volatility)'
        elif beta < 0.8:
            assessment = 'Low Risk (Stable)'
        else:
            assessment = 'Moderate Risk'
        return {'beta': round(beta, 3), 'assessment': assessment}
    except Exception as e:
        return 'Risk data unavailable'


# ─────────────────────────────────────────────────────────────────────────────
# Tool 3 (NEW for Task 16): Price-to-Earnings Ratio
# ─────────────────────────────────────────────────────────────────────────────
def get_pe_ratio(ticker: str):
    """
    Fetches the trailing 12-month Price-to-Earnings (P/E) ratio for a stock.
    The P/E ratio divides the current share price by earnings per share.
    A P/E above the market average (~25) may indicate the stock is overvalued;
    a P/E below 25 may suggest it is undervalued or fairly priced.
    Use this whenever the user asks about valuation, P/E, or whether a stock
    is overvalued or undervalued.
    Args:
        ticker: The stock ticker symbol, e.g. 'AAPL' for Apple.
    Returns:
        A dict with 'ticker', 'trailing_PE' (float or 'N/A'), and
        'valuation_signal' comparing the P/E against the market average of 25.
    """
    print(f'  [TOOL] get_pe_ratio called for: {ticker}')
    try:
        stock = yf.Ticker(ticker)
        pe = stock.info.get('trailingPE', None)

        if pe is None:
            return {'ticker': ticker, 'trailing_PE': 'N/A',
                    'valuation_signal': 'P/E data not available'}

        pe_rounded = round(pe, 2)
        MARKET_AVG_PE = 25.0

        # Compute how far above/below the market average this P/E sits
        diff = pe_rounded - MARKET_AVG_PE
        pct_above = round((diff / MARKET_AVG_PE) * 100, 1)

        if pe_rounded > MARKET_AVG_PE * 1.5:   # more than 50% above market avg
            signal = f'Significantly Overvalued ({pct_above}% above market avg P/E of {MARKET_AVG_PE})'
        elif pe_rounded > MARKET_AVG_PE:        # above market avg
            signal = f'Moderately Overvalued ({pct_above}% above market avg P/E of {MARKET_AVG_PE})'
        elif pe_rounded < MARKET_AVG_PE * 0.75: # more than 25% below market avg
            signal = f'Potentially Undervalued ({abs(pct_above)}% below market avg P/E of {MARKET_AVG_PE})'
        else:
            signal = f'Fairly Valued (close to market avg P/E of {MARKET_AVG_PE})'

        return {
            'ticker'          : ticker,
            'trailing_PE'     : pe_rounded,
            'market_avg_PE'   : MARKET_AVG_PE,
            'valuation_signal': signal,
        }
    except Exception as e:
        return f'Error fetching P/E ratio for {ticker}: {e}'


print('All three tools defined:')
print('  - get_stock_price')
print('  - get_company_risk_score')
print('  - get_pe_ratio  (NEW - Task 16)')

## Section 3 - Initialise the Agent with All Three Tools

We pass the three tool functions directly to `GenerativeModel`. The Gemini SDK
reads their docstrings and type hints to build an internal tool schema.

`enable_automatic_function_calling=True` closes the ReAct loop automatically:
when the model emits a function call, the SDK executes it and feeds the result
back as an observation — no manual round-tripping required.

In [ ]:
tools_list = [get_stock_price, get_company_risk_score, get_pe_ratio]

model = genai.GenerativeModel(
    'gemini-2.5-flash',   # fast, strong tool-use capability
    tools=tools_list
)

# Automatic function calling: SDK handles tool execution and feeds results back
chat = model.start_chat(enable_automatic_function_calling=True)

print('Agent initialised with 3 tools (stock price, risk score, P/E ratio).')

## Section 4 - Required Task: Is Apple Overvalued?

This is the core Task 16 query. The agent must:
1. Identify that a P/E comparison requires calling `get_pe_ratio`
2. Call the tool with `ticker='AAPL'`
3. Receive the P/E value + pre-computed signal
4. Reason about the result against the market average of 25
5. Return a clear, evidence-based valuation opinion

Watch the `[TOOL]` print statements to see the ReAct loop in action.

In [ ]:
query = (
    "Use the get_pe_ratio tool to fetch Apple's current P/E ratio. "
    "The average market P/E is approximately 25. "
    "Based on this comparison, is Apple currently overvalued, undervalued, "
    "or fairly valued? Please explain your reasoning clearly."
)

print(f'USER QUERY: {query}\n')
print('-' * 60)

response = chat.send_message(query)

print('\nAGENT RESPONSE:')
print(response.text)

## Section 5 - Inspect the ReAct Thought Process

By reading `chat.history` we can trace every step of the agent's reasoning:

- **USER** — the original query
- **MODEL + FUNCTION CALL** — the tool the model decided to invoke and its arguments
- **MODEL + FUNCTION RESPONSE** — the data returned by the tool
- **MODEL + TEXT** — the final synthesised answer

This transparency is one of the key advantages of the tool-use paradigm over
pure generation: every factual claim traces back to a concrete data retrieval step.

In [ ]:
print('HISTORY INSPECTION - ReAct Trace\n')
print('=' * 60)

for message in chat.history:
    role = message.role.upper()
    print(f'\n--- {role} ---')
    for part in message.parts:
        if fn := part.function_call:
            print(f'  ACTION    : Called "{fn.name}" with args {dict(fn.args)}')
        elif resp := part.function_response:
            print(f'  OBSERVATION: Tool returned -> {resp.response}')
        elif hasattr(part, 'text') and part.text.strip():
            print(f'  TEXT      : {part.text.strip()}')

print('\n' + '=' * 60)

## Section 6 - Multi-Tool Query: Full Investment Picture for Apple

Now we ask a richer question that forces the agent to combine all three tools.
This demonstrates **multi-step reasoning**: the agent must plan which tools to
call, execute them in sequence (or in parallel if the model supports it), and
synthesise three separate data points into a single coherent recommendation.

We start a fresh chat session so the history is clean and easy to read.

In [ ]:
# Fresh session so the history trace is clean
chat_multi = model.start_chat(enable_automatic_function_calling=True)

query_multi = (
    "I'm evaluating Apple (AAPL) as a potential investment. "
    "Please give me a complete picture using all available tools: "
    "(1) the current stock price, "
    "(2) the risk score based on Beta, and "
    "(3) the P/E ratio compared to the market average of 25. "
    "Based on all three data points together, give me a concise "
    "buy, hold, or avoid recommendation with your reasoning."
)

print(f'USER QUERY: {query_multi}\n')
print('-' * 60)

response_multi = chat_multi.send_message(query_multi)

print('\nAGENT RESPONSE:')
print(response_multi.text)

In [ ]:
# Inspect multi-tool history
print('HISTORY INSPECTION - Multi-Tool Trace\n')
print('=' * 60)

for message in chat_multi.history:
    role = message.role.upper()
    print(f'\n--- {role} ---')
    for part in message.parts:
        if fn := part.function_call:
            print(f'  ACTION    : Called "{fn.name}" with args {dict(fn.args)}')
        elif resp := part.function_response:
            print(f'  OBSERVATION: Tool returned -> {resp.response}')
        elif hasattr(part, 'text') and part.text.strip():
            print(f'  TEXT      : {part.text.strip()}')

print('\n' + '=' * 60)

## Section 7 - Comparative Challenge: MSFT vs NVDA vs AAPL

This cell runs the professor's suggested comparative challenge and extends it
to include the P/E ratio. The agent must fetch all three metrics for all three
companies and identify which is riskiest and which is most overvalued — requiring
six or more tool calls and multi-entity reasoning in a single response.

In [ ]:
chat_compare = model.start_chat(enable_automatic_function_calling=True)

query_compare = (
    "Compare Microsoft (MSFT), NVIDIA (NVDA), and Apple (AAPL) across "
    "all three metrics: current stock price, risk score (Beta), and P/E ratio. "
    "The average market P/E is 25. "
    "Which stock is the riskiest? Which is the most overvalued on a P/E basis? "
    "Present the results as a structured comparison and finish with a "
    "one-sentence verdict for a moderate-risk investor."
)

print(f'USER QUERY: {query_compare}\n')
print('-' * 60)

response_compare = chat_compare.send_message(query_compare)

print('\nAGENT RESPONSE:')
print(response_compare.text)

---
## Section 8 - Reflection: What This Task Reveals About Agentic AI

### The Core Insight: Tools Turn LLMs from Knowers into Actors

A standard LLM without tools can only reason over its training data — it
cannot tell you Apple's stock price *right now* because that information
did not exist when it was trained. Adding `get_pe_ratio` to the agent's
toolkit gives it the ability to reach out, retrieve a current fact, and
incorporate it into a reasoned argument. The LLM transitions from a static
knowledge store to a dynamic reasoning engine connected to live data.

### Why the Docstring Is the Interface

The single most important design decision in this task is writing a clear
docstring for `get_pe_ratio`. Gemini does not receive the function's source
code — it receives a schema derived from the docstring and type hints. A
vague docstring produces incorrect tool invocations; a precise one tells
the model exactly when to call it (`whenever the user asks about valuation
or whether a stock is overvalued`) and what to expect back. This makes
docstrings functional code in agentic systems, not optional documentation.

### The Valuation Signal Design Choice

Rather than returning a raw P/E float, the tool returns a pre-computed
`valuation_signal` string (e.g. `'Significantly Overvalued (120% above
market avg P/E of 25)'`). This is a deliberate agent-design pattern:
*push reasoning down into the tool where it can be precisely controlled,
rather than leaving it entirely to the model's probabilistic generation.*
The model still reasons over the signal, but the numeric threshold logic
is deterministic and auditable in Python — not interpolated from training.

### Limitations of P/E as a Valuation Metric

The agent correctly retrieves and compares P/E ratios, but a sophisticated
investor would note several limitations this tool does not address:
- P/E varies significantly by sector (tech stocks typically trade at higher
  multiples than utilities or financials, so a raw comparison to the
  broad-market average of 25 may be misleading for growth companies).
- Trailing P/E uses past earnings; forward P/E (based on analyst estimates)
  is often more decision-relevant.
- P/E alone says nothing about debt levels, cash flow, or growth runway.

A production-grade agent would add tools for forward P/E, sector-average P/E,
and PEG ratio to provide a more complete valuation picture.

### Conclusion

Task 16 demonstrates the minimal viable extension to make an LLM agent
genuinely useful for a narrow domain: one new tool, one clear docstring,
and a richer prompt that anchors reasoning to a concrete benchmark (P/E = 25).
The ReAct history trace shows that the agent's entire reasoning chain is
transparent and verifiable — a critical property for any AI system used
in a financial context.